# Week 2 — Predicting ABUK Daily Returns with an MLP

This notebook predicts **ABUK's next-session return** from the canonical TradingLab features. It uses a chronological **70% past / 30% future** split—never a random split—then compares smaller, wider, and deeper multilayer perceptrons (MLPs).

We will inspect:

1. training loss versus testing loss across epochs;
2. predicted versus actual returns in both periods;
3. whether more epochs, layers, or hidden neurons actually improve future performance.

In [ ]:
import os
import sys

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tradinglab.data_feed import DataFeed
from tradinglab.features import FEATURE_NAMES, feature_columns
from tradinglab.ml import predict, train_model
from tradinglab.models import DeepMLP

torch.set_num_threads(1)
np.random.seed(7)
torch.manual_seed(7)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Build one stock's supervised-learning table

The features at date `t` are used to predict the return from `t` to `t+1`. Rows with indicator warm-up NaNs are removed, and the final row is removed because its next-session return is unknown.

In [ ]:
SYMBOL = 'ABUK'  # Change this to HRHO or another symbol in feed.symbols.
feed = DataFeed.from_dir('data/egx')
asset = feed.symbols.index(SYMBOL)

X_full = feature_columns(feed, asset)
y_full = np.full(feed.n_days, np.nan)
y_full[:-1] = feed.returns[1:, asset]
valid = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_full)

X = X_full[valid].astype('float32')
y = y_full[valid].astype('float32')
dates = feed.dates[valid]

print(f'Symbol: {SYMBOL}')
print(f'Samples: {len(X):,} | Features: {X.shape[1]}')
print('Features:', FEATURE_NAMES)
print(f'Date range: {dates[0].date()} to {dates[-1].date()}')

## 2. Split 70/30 through time and normalize without leakage

The first 70% is training history and the last 30% is unseen future data. Feature means and standard deviations—and the target scaling—are learned from training rows only, then reused unchanged on the test rows.

In [ ]:
split = int(0.70 * len(X))
X_train_raw, X_test_raw = X[:split], X[split:]
y_train_raw, y_test_raw = y[:split], y[split:]
dates_train, dates_test = dates[:split], dates[split:]

x_mean = X_train_raw.mean(axis=0)
x_std = X_train_raw.std(axis=0)
x_std[x_std < 1e-8] = 1.0
X_train = ((X_train_raw - x_mean) / x_std).astype('float32')
X_test = ((X_test_raw - x_mean) / x_std).astype('float32')

y_mean = float(y_train_raw.mean())
y_std = float(y_train_raw.std())
y_std = y_std if y_std >= 1e-8 else 1.0
y_train = ((y_train_raw - y_mean) / y_std).astype('float32')
y_test = ((y_test_raw - y_mean) / y_std).astype('float32')

print(f'Train: {len(X_train):,} rows ({len(X_train)/len(X):.1%}) | {dates_train[0].date()} → {dates_train[-1].date()}')
print(f'Test:  {len(X_test):,} rows ({len(X_test)/len(X):.1%}) | {dates_test[0].date()} → {dates_test[-1].date()}')
print('Chronology preserved:', dates_train[-1] < dates_test[0])

## 3. Train the baseline MLP

The helper below trains a configurable MLP and converts its standardized predictions back to ordinary daily returns.

In [ ]:
def fit_mlp(hidden=32, layers=2, epochs=200, lr=1e-3, seed=7):
    torch.manual_seed(seed)
    model = DeepMLP(X_train.shape[1], hidden=hidden, n_hidden_layers=layers)
    history = train_model(
        model, X_train, y_train, X_test, y_test, epochs=epochs, lr=lr
    )
    pred_train = predict(model, X_train) * y_std + y_mean
    pred_test = predict(model, X_test) * y_std + y_mean
    return model, history, pred_train, pred_test

baseline_model, baseline_history, pred_train, pred_test = fit_mlp(
    hidden=32, layers=2, epochs=200
)
print(f'Baseline parameters: {sum(p.numel() for p in baseline_model.parameters()):,}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
epochs_axis = np.arange(1, len(baseline_history['train']) + 1)
ax.plot(epochs_axis, baseline_history['train'], label='Training loss', linewidth=1.8)
ax.plot(epochs_axis, baseline_history['test'], label='Testing loss', linewidth=1.8)
best_epoch = int(np.argmin(baseline_history['test']) + 1)
ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.8, label=f'Lowest test loss: epoch {best_epoch}')
ax.set(title=f'{SYMBOL} MLP — Loss by Epoch', xlabel='Epoch', ylabel='MSE on standardized returns')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Predicted returns versus actual returns

Returns are shown in basis points (`1 bp = 0.01%`) so the small daily movements remain readable. The vertical dashed line separates past training data from future testing data.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharey=True)

axes[0].plot(dates_train, y_train_raw * 10_000, label='Actual', linewidth=1.0, alpha=0.8)
axes[0].plot(dates_train, pred_train * 10_000, label='Predicted', linewidth=1.2, alpha=0.85)
axes[0].set_title(f'{SYMBOL} Training Period (Past 70%)')
axes[0].set_ylabel('Next-day return (bp)')
axes[0].legend()

axes[1].plot(dates_test, y_test_raw * 10_000, label='Actual', linewidth=1.0, alpha=0.8)
axes[1].plot(dates_test, pred_test * 10_000, label='Predicted', linewidth=1.2, alpha=0.85)
axes[1].set_title(f'{SYMBOL} Testing Period (Future 30%)')
axes[1].set(xlabel='Date', ylabel='Next-day return (bp)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def regression_metrics(actual, predicted):
    return {
        'MSE': float(np.mean((actual - predicted) ** 2)),
        'MAE': float(np.mean(np.abs(actual - predicted))),
        'Directional accuracy': float(np.mean(np.sign(actual) == np.sign(predicted))),
        'Correlation': float(np.corrcoef(actual, predicted)[0, 1]),
    }

baseline_metrics = pd.DataFrame({
    'Train': regression_metrics(y_train_raw, pred_train),
    'Test': regression_metrics(y_test_raw, pred_test),
})
display(baseline_metrics.style.format('{:.6f}'))

## 5. Experiment: epochs, width, and depth

Each candidate sees the same training rows, test rows, normalization, learning rate, and random seed. Only network size and training duration change. This makes the comparison understandable—but because we inspect test performance repeatedly, this is an **exploratory test set**, not a final untouched validation set.

In [ ]:
experiments = [
    {'name': 'Small / short', 'hidden': 16, 'layers': 1, 'epochs': 100},
    {'name': 'Baseline', 'hidden': 32, 'layers': 2, 'epochs': 200},
    {'name': 'Wider', 'hidden': 128, 'layers': 2, 'epochs': 300},
    {'name': 'Deeper', 'hidden': 64, 'layers': 4, 'epochs': 300},
]

experiment_rows = []
experiment_histories = {}
for config in experiments:
    model, history, train_prediction, test_prediction = fit_mlp(
        hidden=config['hidden'],
        layers=config['layers'],
        epochs=config['epochs'],
    )
    train_stats = regression_metrics(y_train_raw, train_prediction)
    test_stats = regression_metrics(y_test_raw, test_prediction)
    experiment_histories[config['name']] = history
    experiment_rows.append({
        **config,
        'parameters': sum(p.numel() for p in model.parameters()),
        'best epoch': int(np.argmin(history['test']) + 1),
        'train MSE': train_stats['MSE'],
        'test MSE': test_stats['MSE'],
        'test direction': test_stats['Directional accuracy'],
        'test correlation': test_stats['Correlation'],
    })

results = pd.DataFrame(experiment_rows).set_index('name').sort_values('test MSE')
display(results.style.format({
    'train MSE': '{:.8f}', 'test MSE': '{:.8f}',
    'test direction': '{:.2%}', 'test correlation': '{:.4f}',
    'parameters': '{:,.0f}',
}))

In [ ]:
ordered = results.sort_values('parameters')
x = np.arange(len(ordered))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x - width/2, ordered['train MSE'] * 1e6, width, label='Train MSE')
axes[0].bar(x + width/2, ordered['test MSE'] * 1e6, width, label='Test MSE')
axes[0].set_xticks(x, ordered.index, rotation=15)
axes[0].set(title='Raw-return Error by Model Size', ylabel='MSE × 1,000,000')
axes[0].legend()

for name, history in experiment_histories.items():
    axes[1].plot(history['test'], label=name, linewidth=1.3)
axes[1].set(title='Testing Loss Across Epochs', xlabel='Epoch', ylabel='Standardized-return MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_name = results['test MSE'].idxmin()
largest_name = results['parameters'].idxmax()
best = results.loc[best_name]
largest = results.loc[largest_name]

print(f'Best future MSE: {best_name} ({best["test MSE"]:.8f})')
print(f'Largest model:   {largest_name} ({largest["parameters"]:,.0f} parameters, test MSE {largest["test MSE"]:.8f})')
if best_name == largest_name:
    print('In this run the largest model won, but one result is not evidence that bigger is always better.')
else:
    print('Bigger did not mean better here: a smaller model generalized better to the future period.')
print('The honest next step is to choose a design here, then evaluate it once on a separate untouched validation period.')

## Takeaway

A larger network has more capacity to fit the past, but daily stock returns are noisy and the dataset is small. More neurons, layers, or epochs can reduce training loss while leaving future loss unchanged—or making it worse. **Bigger does not automatically mean better.** Choose complexity using future-period evidence, watch the train/test gap, and reserve a final validation period before making a performance claim.